# Apparent orders, apparent barriers, and lumped rate laws

Experimentalists characterize a catalyst by **apparent reaction orders** and an **apparent activation energy** — the local power-law/Arrhenius fit of the overall rate. Both are logarithmic sensitivities of the steady-state turnover frequency, computed here analytically by the same implicit differentiation used for the degree of rate control:

- apparent order  `n_i = d ln r / d ln P_i`
- apparent barrier `E_a^app = R T² d ln r / dT`

Because the derivatives propagate through the steady-state coverages, these are the true lumped-kinetics descriptors, not the bare elementary-step values.

In [1]:
import discopt.mkm as mk
from discopt.mkm.examples import co_oxidation
from discopt.mkm.analysis import apparent_orders, apparent_activation_energy
m, reactor = co_oxidation(T=500.0)
m

#,reaction,kinetics,type
1,CO + Pt ⇌ CO∗,"A=10000, Ea=0",reversible
2,O2 + 2 Pt ⇌ 2 O∗,"A=10000, Ea=0",reversible
3,CO∗ + O∗ ⇌ CO2 + 2 Pt,"A=1e+08, Ea=0.7",reversible


In [2]:
CO, O2, CO2 = m._by_name['CO'], m._by_name['O2'], m._by_name['CO2']
sol = mk.solve_steady_state(m, reactor)

## Apparent reaction orders (differential reactor)

In [3]:
for g, n in apparent_orders(sol, CO2).items():
    print(f'  order in {g.name:3s} = {n:+.3f}')
# CO order is negative here: CO inhibits its own oxidation (coverage effect)

  order in CO  = -0.171
  order in O2  = +0.269


## Apparent activation energy

In [4]:
Ea_app = apparent_activation_energy(sol, CO2)
print(f'apparent Ea = {Ea_app:.3f} eV   (elementary surface barrier was 0.7 eV)')
# the apparent barrier differs from any single elementary barrier

apparent Ea = 0.675 eV   (elementary surface barrier was 0.7 eV)


## Overall (lumped) rate expression

For a quasi-equilibrium mechanism the coverages solve in closed form, so the overall rate can be derived symbolically (SymPy) — the classic Langmuir-Hinshelwood result. For a general network there is no closed form; the numerical model *is* the lumped rate, and the apparent orders/barrier above characterize it locally.

In [5]:
# A + * <=> A* (quasi-equilibrium),  A* -> B + * (rate-determining)
lh = mk.Model('lh', T=500, R=8.314)
s = lh.site('s', density=1.0)
A, B = lh.gas('A'), lh.gas('B')
As = lh.adsorbate('A*', site=s)
lh.step(A + s >> As, Keq=5.0, equilibrated=True)
lh.step(As >> B + s, kf=3.0, irreversible=True)
rate, syms = mk.lumped_rate_expression(lh, B)
rate   # SymPy renders the Langmuir-Hinshelwood rate

Keq0*P_A*kf1/(Keq0*P_A + 1)